# Dr Amato - Orbital Mech Tutorials, Tutorial 1
Lecture 5 - Coordinate Systems, Orbital Elements and Orbit Propagation

## Question 1

### Part a

In [ ]:
# Variable loading
import numpy as np

# Initialising the variable
r = np.array([
    1.073,
    1.514,
    0.541
])

# WIP

# Calculating the RA
RA = np.atan(r[1] / r[0]) * 180/np.pi

# Want to check the quadrant state
#   Quadrant 1, add 0 degrees
if r[0] > 0 and r[1] > 0:
    RA += 0

#   Quadrant 2, add 90 degrees
elif r[0] < 0 and r[1] > 0:
    RA += 90

#   Quadrant 3, add 180 degrees
elif r[0] < 0 and r[1] < 0:
    RA += 180

#   Quadrant 4, add 270 degrees
elif r[0] > 0 and r[1] < 0:
    print("False")



RA = np.atan(r[1] / r[0]) * 180/np.pi
print(RA)

# Calculating the DE
xylength = np.sqrt(r[0]**2 + r[1]**2)
DE = np.atan(r[2] / xylength) * 180/np.pi
print(DE)

True
54.674089143953495
16.253427217902644


### Part b

Solution consists of a rotation matrix in the x axis

### Part c

In [8]:
import numpy as np

# Initialising the variable
r = np.array([
    1.073,
    1.514,
    0.541
])

# Obliquity value
varepsilon = 23.4
#   Converted to radians
varepsilon *= np.pi/180

# Rotationg from ECI to ECIE
ECI_to_ECIE = np.array([
    [1, 0, 0,],
    [0, np.cos(varepsilon), np.sin(varepsilon)],
    [0, -np.sin(varepsilon), np.cos(varepsilon)],
])

# Position vector of asteroid in ECIE frame
r_ECIE = np.linalg.matmul(ECI_to_ECIE, r)
print("Rotated value = ", r_ECIE)

Rotated value =  [ 1.073       1.60433751 -0.10477665]


## Question 2

Solution is a written proof, will be written down later

## Question 3

Solution is a written proof, will be written down later

## Question 4

In [27]:
import numpy as np
import time
import requests

# Change option from a, b or c for your respective part
opt = "a"

# Solving part a, b and c
if opt == "a":
    y, m, d = 2008, 1, 1
    UT = 12

    # Positive as it is due eastward
    Lambda1, Lambda2 = 18, 3

elif opt == "b":
    y, m, d = 2005, 7, 4
    UT = 20

    # Negative due to being westward
    Lambda1, Lambda2 = -118, -15

elif opt == "c":
    # This option gets current time and location of where you are based when you executed this code block
    print("Current date:")
    print(time.strftime("%Y-%m-%d %H:%M:%S"), "\n")

    # Gets the year, month and day of current time
    y, m, d = int(time.strftime("%Y")), int(time.strftime("%m")), int(time.strftime("%d"))
    # Gets current time and converts into decimal value
    UT = int(time.strftime("%H")) + int(time.strftime("%M")) / 60 + int(time.strftime("%S")) / 3600

    # Uses requests to get the longitude of place
    # Will require an internet connection so if there is no internet, then will default to Greenwhich (longitude 0)
    try:

        # Collects information using ipinfo.io
        response = requests.get("https://ipinfo.io/json")
        data = response.json()

        # Since we have the lontitude as a decimal, we do not need Lamdba2
        lattitude, Lambda1 = data["loc"].split(",")
        Lambda1 = float(Lambda1)
        Lambda2 = 0

        # No interest in lattitude so we disregard lattitude
        latittude = None

    except:
        print("Error, cannot connect to ipinfo.io")
        print("Will default to the same Greenwhich position (0 longitude)")
        Lambda1 = 0
        Lambda2 = 0

Lambda = Lambda1 + Lambda2/60

## J0
JD = 367 * y - round((7 * (y + round((m+9)/12) ) ) / (4)) + round(275 * m / 9) + d + 1721013.5 + UT/24

## T0
T0 = (JD - 2451545) / 36525
print(T0)

## theta G0
theta_G0 = 100.4606184 + 36000.77004 * T0 + 0.000387933 * T0**2 - 2.583e-8  * T0**3
print(theta_G0)

## theta G
theta_G =  theta_G0 + 360.98564724 * UT/24
while theta_G > 360:
    theta_G -= 360

print(theta_G)

## Local Sideral Time
LocalSiderealTime = theta_G + Lambda
print(LocalSiderealTime)


0.07997262149212868
2979.536576715145
280.02940033514506
298.07940033514507


## Question 5

### part a

In [39]:
import time
import requests

def gettheta_G():
    print("Current date:")
    print(time.strftime("%Y-%m-%d %H:%M:%S"), "\n")

    y, m, d = int(time.strftime("%Y")), int(time.strftime("%m")), int(time.strftime("%d"))
    UT = int(time.strftime("%H")) + int(time.strftime("%M")) / 60 + int(time.strftime("%S")) / 3600

    try:
        response = requests.get("https://ipinfo.io/json")
        data = response.json()
        lattitude, Lambda1 = data["loc"].split(",")
        Lambda1 = float(Lambda1)
        Lambda2 = 0

        latittude = None

    except:
        print("Error, cannot connect to ipinfo.io")
        print("Will default to the same Greenwhich position (0 longitude)")
        Lambda1 = 0
        Lambda2 = 0

    Lambda = Lambda1 + Lambda2/60

    JD = 367 * y - round((7 * (y + round((m+9)/12) ) ) / (4)) + round(275 * m / 9) + d + 1721013.5 + UT/24
    T0 = (JD - 2451545) / 36525
    theta_G0 = 100.4606184 + 36000.77004 * T0 + 0.000387933 * T0**2 - 2.583e-8  * T0**3
    theta_G =  theta_G0 + 360.98564724 * UT/24
    while theta_G > 360:
        theta_G -= 360

    print("Theta_G = ", theta_G, "degrees")
    return theta_G * np.pi/180

# Get the theta_G of the current time and place
theta_G = gettheta_G()

# Apply matrix transformation from ECEF to ECI
ECI_to_ECEF = np.array([
    [np.cos(theta_G), np.sin(theta_G), 0],
    [-np.sin(theta_G), np.cos(theta_G), 0],
    [0, 0, 1]
])
ECEF_to_ECI = np.linalg.inv(ECI_to_ECEF)
print(ECEF_to_ECI)

# Getting current r_ECEF
# Want to get the current lattitude and longitude angles
try:
    response = requests.get("https://ipinfo.io/json")
    data = response.json()
    lattitude, longitude = data["loc"].split(",")
    lattitude, longitude = float(lattitude), float(longitude)

except:
    print("Error, cannot connect to ipinfo.io")
    print("Will default to the same Greenwhich position (0 longitude, 51.58 lattitude)")
    longitude = 0
    lattitude = 51.58

# Convert lattitude and longitude from degrees to radians
lattitude *= np.pi/180
longitude *= np.pi/180

# Make the assumption of r as 6370, and then get the r_ECEF
r = 6370
r_ECEF = np.array([
    r * np.sin(lattitude) * np.cos(longitude),
    r * np.sin(lattitude) * np.sin(longitude),
    r * np.cos(lattitude)
])
print("r_ECEF coordinates:")
print(r_ECEF)

# Apply matrix transformation to get your ECI frame of your current position
r_ECI = np.linalg.matmul(ECEF_to_ECI, r_ECEF)
print("r_ECI: \n", r_ECI)

Current date:
2026-08-26 10:03:22 

Theta_G =  125.92693030609007 degrees
[[-0.58675303 -0.80976594 -0.        ]
 [ 0.80976594 -0.58675303  0.        ]
 [ 0.          0.          1.        ]]
r_ECEF coordinates:
[4.99052432e+03 2.14268672e+00 3.95873243e+03]
r_ECI: 
 [-2929.94033763  4039.89940032  3958.73243011]
